# 02 — Classical baselines

Brute force, greedy top-K (Sharpe), Markowitz-then-round, and simulated
annealing — all on the same cardinality-constrained QUBO. Brute force is
the ground truth as long as `n` is small enough.

Methodology: SA is QAOA's real classical analogue, so we report 10 seeds
(median + IQR) for it just like we will for QAOA.

In [ ]:
# === Bootstrap (Colab + local) ===
import sys, os
try:
    import google.colab  # noqa: F401
    get_ipython().system('test -d /content/fys5419 || git clone -q https://github.com/egil10/fys5419.git /content/fys5419')
    get_ipython().run_line_magic('cd', '/content/fys5419/project2/code/notebooks')
except ImportError:
    pass
sys.path.append('..')
from scripts.colab import setup; setup()

# === Project imports ===
import numpy as np
import pandas as pd

from scripts.data      import load_returns
from scripts.baskets   import config
from scripts.portfolio import PortfolioProblem
from scripts.classical import (
    brute_force, greedy_top_k, markowitz_round, simulated_annealing,
)
from scripts.metrics   import sharpe

### Problem instance — `n=8, K=2` (fixed)

In [ ]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN',
           'NVDA', 'IBM',  'HON',   'ACN']
K, LAM, AP = 2, 2.0, 0.5
START, END = '2023-01-01', '2025-12-31'

r  = load_returns(TICKERS, START, END, cache_name='n8_universe')
pf = PortfolioProblem(r.mu, r.Sigma, lam=LAM, A=AP, K=K, tickers=tuple(r.tickers))
print(pf)

### Deterministic baselines (single run)

In [ ]:
det = {
    'brute_force':     brute_force(pf),
    'greedy_sharpe':   greedy_top_k(pf, score='sharpe'),
    'markowitz_round': markowitz_round(pf),
}
rows = [{
    'solver':    sr.name,
    'bitstring': sr.bitstring,
    'tickers':   '+'.join(sr.tickers(pf)),
    'cost':      sr.cost,
    'feasible':  sr.feasible,
    'sharpe':    sharpe(sr.x, *pf.annualised() if False else (pf.mu, pf.Sigma)),
    'runtime_s': sr.runtime,
} for sr in det.values()]
pd.DataFrame(rows).sort_values('cost')

### Simulated annealing — 10 seeds, report median + IQR

In [ ]:
N_SEEDS = 10
sa_runs = [simulated_annealing(pf, n_sweeps=1000, seed=s) for s in range(N_SEEDS)]
sa_costs = np.array([sr.cost for sr in sa_runs])
best_sa  = min(sa_runs, key=lambda sr: sr.cost)

q25, q50, q75 = np.percentile(sa_costs, [25, 50, 75])
print(f'SA over {N_SEEDS} seeds:')
print(f'  median cost: {q50:.6f}  (IQR {q25:.6f} -> {q75:.6f})')
print(f'  best  cost:  {best_sa.cost:.6f}  ({best_sa.bitstring})')
print(f'  brute force: {det["brute_force"].cost:.6f}')